In [1]:
from bloqade.gemini import logical as gemini_logical

In [2]:
from bloqade import squin

In [3]:
import math

In [4]:
THETA = math.pi / 16

In [5]:
@gemini_logical.kernel(aggressive_unroll=True, verify=False)
def star_on_plus_kernel():
    reg = squin.qalloc(1)
    squin.h(reg[0])
    gemini_logical.star_rz(THETA, reg[0])

In [6]:
from bloqade.gemini import GeminiLogicalSimulator

In [7]:
task = GeminiLogicalSimulator().task(logical_kernel=star_on_plus_kernel)

In [8]:
task.noiseless_tsim_circuit.diagram()

In [9]:
import numpy as np

from bloqade.gemini.logical.stdlib import default_post_processing

SHOTS = 1_000_000


@gemini_logical.kernel(aggressive_unroll=True, verify=False)
def star_on_plus_kernel_x():
    reg = squin.qalloc(2)
    squin.h(reg[0])
    gemini_logical.star_rz(THETA, reg[0])
    
    squin.h(reg[1])
    squin.cx(reg[1], reg[0])
    squin.h(reg[1])

    squin.h(reg[0])
    return default_post_processing(reg)


@gemini_logical.kernel(aggressive_unroll=True, verify=False)
def star_on_plus_kernel_y():
    reg = squin.qalloc(2)
    squin.h(reg[0])
    gemini_logical.star_rz(THETA, reg[0])

    squin.h(reg[1])
    squin.cx(reg[1], reg[0])
    squin.h(reg[1])

    squin.sqrt_z(reg[0])
    squin.h(reg[0])
    return default_post_processing(reg)


@gemini_logical.kernel(aggressive_unroll=True, verify=False)
def star_on_plus_kernel_z():
    reg = squin.qalloc(2)
    squin.h(reg[0])
    gemini_logical.star_rz(THETA, reg[0])

    squin.h(reg[1])
    squin.cx(reg[1], reg[0])
    squin.h(reg[1])
    
    return default_post_processing(reg)


In [10]:
def postselected_observable_bits(result):
    detectors = np.asarray(result.detectors, dtype=bool)
    observables = np.asarray(result.observables, dtype=bool)

    print(f"detectors.shape: {detectors.shape}")
    print(f"detectors[:, -3:].shape: {detectors[:, -3:].shape}")
    print(f"observables.shape: {observables.shape}")
    accepted = ~np.any(detectors[:, -3:], axis=1)
    if not np.any(accepted):
        raise ValueError("no accepted shots after detector postselection")

    return observables[accepted, 0].astype(np.uint8), accepted


x_task = GeminiLogicalSimulator().task(logical_kernel=star_on_plus_kernel_x)
y_task = GeminiLogicalSimulator().task(logical_kernel=star_on_plus_kernel_y)
z_task = GeminiLogicalSimulator().task(logical_kernel=star_on_plus_kernel_z)

x_shots = x_task.run(shots=SHOTS)
y_shots = y_task.run(shots=SHOTS)
z_shots = z_task.run(shots=SHOTS)

x_shots_arr, x_accepted = postselected_observable_bits(x_shots)
y_shots_arr, y_accepted = postselected_observable_bits(y_shots)
z_shots_arr, z_accepted = postselected_observable_bits(z_shots)

print(f"accepted X shots: {len(x_shots_arr)}/{SHOTS} ({np.mean(x_accepted):.6f})")
print(f"accepted Y shots: {len(y_shots_arr)}/{SHOTS} ({np.mean(y_accepted):.6f})")
print(f"accepted Z shots: {len(z_shots_arr)}/{SHOTS} ({np.mean(z_accepted):.6f})")


E0529 11:41:09.677655 8465985 slow_operation_alarm.cc:73] Constant folding an instruction is taking > 1s:

  %gather.106 = s32[9000009,1,4]{2,1,0} gather(%constant.1391, %broadcast.1388), offset_dims={1,2}, collapsed_slice_dims={}, start_index_map={0}, index_vector_dim=1, slice_sizes={1,4}, metadata={op_name="jit(_sample_component_jit)/jit(evaluate)/gather" stack_frame_id=255}

This isn't necessarily a bug; constant-folding is inherently a trade-off between compilation time and speed at runtime. XLA has some guards that attempt to keep constant folding from taking too long, but fundamentally you'll always be able to come up with an input program that takes a long time.

If you'd like to file a bug, run with envvar XLA_FLAGS=--xla_dump_to=/tmp/foo and attach the results.
E0529 11:41:09.931562 8465980 slow_operation_alarm.cc:140] The operation took 1.258755s
Constant folding an instruction is taking > 1s:

  %gather.106 = s32[9000009,1,4]{2,1,0} gather(%constant.1391, %broadcast.1388), o

detectors.shape: (1000000, 6)
detectors[:, -3:].shape: (1000000, 3)
observables.shape: (1000000, 2)
detectors.shape: (1000000, 6)
detectors[:, -3:].shape: (1000000, 3)
observables.shape: (1000000, 2)
detectors.shape: (1000000, 6)
detectors[:, -3:].shape: (1000000, 3)
observables.shape: (1000000, 2)
accepted X shots: 475720/1000000 (0.475720)
accepted Y shots: 475866/1000000 (0.475866)
accepted Z shots: 476087/1000000 (0.476087)


In [11]:
import sys
from pathlib import Path

demo_dir = Path.cwd() if (Path.cwd() / "stdlib").exists() else Path.cwd() / "demo"
if str(demo_dir) not in sys.path:
    sys.path.append(str(demo_dir))

from stdlib.tomography import fidelity_from_counts

star_bloch_vector = np.array((math.cos(THETA), math.sin(THETA), 0.0))
fidelity_estimate = fidelity_from_counts(
    x_shots_arr,
    y_shots_arr,
    z_shots_arr,
    binary_precision=None,
    target_bloch=star_bloch_vector,
)

fidelity_from_zero_one_counts, ex: 0.9175060960228706, ey: -0.0008699928131028483, ez: 0.001195159708204593


In [12]:
star_bloch_vector

array([0.98078528, 0.19509032, 0.        ])

In [13]:
fidelity_estimate

{'point': 0.9498533732407022,
 'median': 0.9498533732407022,
 'low': 0.9492336960806529,
 'high': 0.9504730504007515,
 'error': 0.0006196771600492533,
 'bloch': (0.9175060960228706, -0.0008699928131028483, 0.001195159708204593)}